# Recurrent Neural Network — Lecture 27.03.2026

![RNN on the board](images/rnn_board_2026-03-27.png)

## Network Architecture

The diagram shows a **discrete-time recurrent neural network (RNN)** with 4 neurons arranged in a diamond topology:

| Neuron | Role | Self-loop |
|--------|------|-----------|
| 0 (left) | Input — receives $X(t)$ | No |
| 1 (top) | Hidden, recurrent | **Yes** |
| 2 (bottom) | Hidden, recurrent | **Yes** |
| 3 (right) | Output — produces $Y(t)$ | No |

### Connections

* **Input → Hidden:** neuron 0 feeds neurons 1 and 2
* **Hidden ↔ Hidden (recurrent):** neurons 1 and 2 have self-loops **and** cross-connections to each other (fully connected recurrent hidden layer)
* **Hidden → Output:** neurons 1 and 2 both feed neuron 3

## Weight Matrix

The weight matrix $W$ where $w_{ij}$ denotes the weight of the connection **from neuron $j$ to neuron $i$**:

$$
W = \begin{pmatrix}
0 & 0 & 0 & 0 \\
w_{10} & w_{11} & w_{12} & 0 \\
w_{20} & w_{21} & w_{22} & 0 \\
0 & w_{31} & w_{32} & 0
\end{pmatrix}
$$

* $w_{10},\; w_{20}$ — input weights (neuron 0 → hidden neurons 1, 2)
* $w_{11},\; w_{22}$ — self-recurrent weights (self-loops)
* $w_{12},\; w_{21}$ — cross-recurrent weights between hidden neurons
* $w_{31},\; w_{32}$ — output weights (hidden neurons 1, 2 → neuron 3)

Total **8 non-zero trainable weights** in $W$: $w_{10}, w_{11}, w_{12}, w_{20}, w_{21}, w_{22}, w_{31}, w_{32}$.

## RNN Dynamics (discrete time)

At each time step $t$, neuron outputs are computed as follows.

### Hidden layer (neurons 1 and 2) — use state from the **previous** step $t-1$:

$$
y_1(t) = f\!\Big(w_{10}\, x(t) \;+\; w_{11}\, y_1(t-1) \;+\; w_{12}\, y_2(t-1)\Big)
$$

$$
y_2(t) = f\!\Big(w_{20}\, x(t) \;+\; w_{21}\, y_1(t-1) \;+\; w_{22}\, y_2(t-1)\Big)
$$

### Output layer (neuron 3) — uses the **current** hidden state:

$$
Y(t) = f\!\Big(w_{31}\, y_1(t) \;+\; w_{32}\, y_2(t)\Big)
$$

Here $f(\cdot)$ is the activation function (e.g. $\tanh$, sigmoid, etc.) and $x(t) \equiv X(t)$ is the external input at time $t$.

### In matrix form

Denoting $\mathbf{h}(t) = \big(y_1(t),\; y_2(t)\big)^T$:

$$
\mathbf{h}(t) = f\!\left(
\underbrace{\begin{pmatrix} w_{11} & w_{12} \\ w_{21} & w_{22} \end{pmatrix}}_{W_h}
\mathbf{h}(t-1)
\;+\;
\underbrace{\begin{pmatrix} w_{10} \\ w_{20} \end{pmatrix}}_{W_x}
x(t)
\right)
$$

$$
Y(t) = f\!\left(
\underbrace{\begin{pmatrix} w_{31} & w_{32} \end{pmatrix}}_{W_y}
\mathbf{h}(t)
\right)
$$

## Loss Function (from the board)

Let the training set be

$$
\mathcal{D} = \left\{\left(X^{(p)}(t),\, \sigma^{(p)}(t)\right)\right\},\quad t=1,\dots,T,\; p=1,\dots,P.
$$

### 1) Seq-to-Vec

Average loss over patterns:

$$
E = \frac{1}{P}\sum_{p=1}^{P} E^{(p)}.
$$

For one pattern (using final time $T$):

$$
E^{(p)} = \sum_k \frac{1}{2}\left(y_k^{(p)}(T) - \sigma_k^{(p)}\right)^2,
$$

and training minimizes $E$ with respect to network parameters (weights).

### 2) Seq-to-Seq

For one pattern, error is accumulated over all time steps:

$$
E^{(p)} = \sum_{t=1}^{T}\sum_k \left(y_k^{(p)}(t) - \sigma_k^{(p)}(t)\right)^2.
$$